# Stage 4.03 — seed-999 end-to-end smoke
Runs four Native smoke episodes outside the analysis matrix. Scientific failure is allowed; infrastructure or provenance failure is not.

In [ ]:
import csv,os,subprocess
from pathlib import Path
GPU="0"  # change to one idle physical A100
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; N=Path.home()/"stage1-native"; OFT=Path.home()/"openvla-oft"; PY=Path.home()/"venv-stage4-openvla/bin/python"; MAIN=Path.home()/"stage4"; OUT=Path.home()/"stage4_smoke"; OUT.mkdir(exist_ok=True); MAN=OUT/"stage4_smoke_manifest.csv"; AUDIT=OUT/"stage4_smoke_pairing_audit.csv"; SNAP=Path((MAIN/"stage4_checkpoint_snapshot.txt").read_text().strip())
state=subprocess.run(["nvidia-smi","-i",GPU,"--query-gpu=memory.used,utilization.gpu","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); print("Physical GPU",GPU,"state:",state); used,util=[int(x.strip()) for x in state.split(',')]
if used>=500 or util>=5: raise SystemExit(f"STOP: physical GPU {GPU} is not idle: {state}")
bench=subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(); plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.make_stage4_smoke_manifest","--output",str(MAN),"--git-sha",bench,"--libero-plus-git-sha",plus],cwd=R,check=True)
base=os.environ.copy(); base.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONPATH":str(OFT)+os.pathsep+str(R)})
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage4.yaml"),"--manifest",str(MAN),"--scene","id","--expected-rows","4","--expected-cells-per-key","1","--audit-output",str(AUDIT)],cwd=R,env=base,check=True)
ood=base.copy(); ood.update({"PYTHONPATH":str(P)+os.pathsep+str(OFT)+os.pathsep+str(R),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+ood.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+ood.get("LD_LIBRARY_PATH","")})
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage4.yaml"),"--manifest",str(MAN),"--scene","ood","--expected-rows","4","--expected-cells-per-key","1","--audit-output",str(AUDIT)],cwd=R,env=ood,check=True)
for scene,env in (("id",base),("ood",ood)):
    cmd=[str(PY),"-u","-m","async_vla_benchmark.scripts.run_stage4","--config",str(R/"async_vla_benchmark/configs/stage4.yaml"),"--manifest",str(MAN),"--output-dir",str(OUT),"--scene",scene,"--openvla-oft-checkout",str(OFT),"--checkpoint-snapshot",str(SNAP),"--resume","--verbose"]
    subprocess.run(cmd,cwd=R,env=env,check=True)
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.validate_stage4_smoke","--manifest",str(MAN),"--output-dir",str(OUT)],cwd=R,env=base,check=True)
print("STOP HERE: paste the Stage 4 smoke PASS line before notebook 04")